# Customer - Bronze

**Reads:** `bronze.bronze_customers`  
**Writes:** `validate.bronze_clean_customers` (overwrite)  
**Parameter:** `AS_OF`, the reference date used for `age` and `tenure`.

## Stakeholders Requests
- Trim and capitalize the customers names and create new attribute for Full Name.
- Need Customers Data in Case as per business entity (Such as PAN -> UPPERCASE, email -> LOWERCASE,customer_id -> UPPERCASE, etc.).
- Standardize category attribute (such as gender -> M/F, is_active -> True/False, Country -> IN).
- Standardize Date Format in all date attribute.
- Parse annual income and phone number.
- New Attributes to add in DF as per product team requests.
    1) full_name as referred in 1st point.
    2) age
    3) tenure
    3) email domain
    4) pan_holder_type (such as 'COMPANY' or 'INDIVIDUAL')
    5) income_band (Such as UNKNOWN, MASS, AFFLUENT, HNI, ULTRA_HNI)
- Finally remove multiple customers updates and keep only latest customer updates using 'customer_id' & 'updated_at' as key

## Imports

In [ ]:
# Imports
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import StructField, StructType

## Shared functions

In [ ]:
%run NB_FUNCTIONS

## Parameters

The next cell is tagged `parameters`, so a pipeline can override it at run time.

In [ ]:
AS_OF = ""

## Reading Dataframe

In [ ]:
cust_df = spark.read.table("bronze.bronze_customers")

## Removing Null Tokens from Dataframe

In [ ]:
# Removing null tokens
cust_n = normalise_null(cust_df)

## Trimming, Capitalization and Casing

In [ ]:
# Trim and Casing
cust_c = (
    cust_n.withColumn("first_name", F.initcap(F.trim("first_name")))
    .withColumn("last_name", F.initcap(F.trim("last_name")))
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("pan", F.upper(F.trim("pan")))
    .withColumn("city", F.initcap(F.trim("city")))
    .withColumn("customer_id", F.upper(F.trim("customer_id")))
)

## Categorizing Gender & Country

In [ ]:
GENDER = {
    "M": "M",
    "MALE": "M",
    "F": "F",
    "FEMALE": "F",
    "X": "O",
    "OTHER": "O",
    "O": "O",
}

COUNTRY = {"INDIA": "IN", "IN": "IN", "IND": "IN"}

cust_c = (
    cust_c.withColumn("gender", mapping_values("gender", GENDER, "U"))
    .withColumn("country", mapping_values("country", COUNTRY, "UNKNOWN"))
    .withColumn("kyc_status", F.upper(F.trim("kyc_status")))
    .withColumn("risk_rating", F.upper(F.trim("risk_rating")))
)

## Standardizing Boolean Values

In [ ]:
BOOL = {
    "Y": True,
    "YES": True,
    "1": True,
    "TRUE": True,
    "N": False,
    "NO": False,
    "0": False,
    "FALSE": False,
}
cust_c = cust_c.withColumn("is_active", mapping_values("is_active", BOOL))

## Standardizing Dates

In [ ]:
DATE_FORMAT = ["yyyy-MM-dd", "dd/MM/yyyy", "dd-MMM-yyyy", "yyyy/MM/dd"]

cust_c = (
    cust_c.withColumn("dob_raw", F.col("dob"))  # keep original for DQ
    .withColumn("dob", date_parse("dob", DATE_FORMAT))
    .withColumn("customer_since", date_parse("customer_since", DATE_FORMAT))
    .withColumn("updated_at", F.to_timestamp("updated_at", "yyyy-MM-dd HH:mm:ss"))
)

## Parsing Amount

In [ ]:
cust_c = cust_c.withColumn("raw_income", F.col("annual_income")).withColumn(
    "annual_income", amount_parse("annual_income")
)

## Parsing Phone Number

In [ ]:
digits = F.regexp_replace("phone", r"\D", "")  # keep digits only
cust_c = cust_c.withColumn("phone_raw", F.col("phone")).withColumn(
    "phone",
    F.nullif(F.regexp_extract(digits, r"^(?:91|0)?([6-9]\d{9})$", 1), F.lit("")),
)

## Keeping Only latest values as per business key

In [ ]:
cust_l = latest_by_key(cust_c.dropDuplicates(), ["customer_id"], ["updated_at"])

## User's request to add new attributes!

In [ ]:
# Current Date (Dynamic Variable passed through pipeline)

cust_l = (
    cust_l.withColumn("full_name", F.concat_ws(" ", "first_name", "last_name"))
    .withColumn(
        "age", F.floor(F.months_between(F.lit(AS_OF), F.col("dob")) / 12).cast("int")
    )
    .withColumn("tenure", F.round(F.months_between(F.lit(AS_OF), F.col("dob")) / 12, 1))
    .withColumn("email_domain", F.regexp_extract("email", r"@(.+)$", 1))
    .withColumn(
        "pan_holder_type",
        F.when(F.substring("pan", 4, 1) == "P", "INDIVIDUAL")
        .when(F.substring("pan", 4, 1) == "C", "COMPANY")
        .otherwise("OTHER"),
    )
    .withColumn(
        "income_band",
        F.when(F.col("annual_income").isNull(), "UNKNOWN")
        .when(F.col("annual_income") < 500000, "MASS")
        .when(F.col("annual_income") < 1500000, "AFFLUENT")
        .when(F.col("annual_income") < 5000000, "HNI")
        .otherwise("ULTRA_HNI"),
    )
)

## Processing to Silver Layer

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS validate")
cust_l.write.mode("overwrite").format("delta").saveAsTable(
    "validate.bronze_clean_customers"
)